In [3]:
from pathlib import Path
import sys
import urllib
import importlib

from sqlalchemy import create_engine

project_path = Path(r"C:\Users\kaszi\Projekty moje\Projekt zespołowy sem 7")

if str(project_path) not in sys.path:
    sys.path.append(str(project_path))

server = r".\OPTIMA"
db = "ndr_analysis"
driver = "ODBC Driver 17 for SQL Server"

params = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={db};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

from final_to_app import reports

importlib.reload(reports)

print("IMPORT OK")

data = reports.get_report_data(engine)

print("\nDATA:")
for name, table in data.items():
    print(name, table.shape)

report_json = reports.get_report_json(engine)

print("\nJSON:")
print(type(report_json))
print(report_json.keys())
print(report_json["kpi"][:2])

table_path = project_path / "outputs" / "test_tables"
chart_path = project_path / "outputs" / "test_charts"

reports.export_report_tables(data, table_path)
charts = reports.export_report_charts(data, chart_path)

print("\nTABLES:")
for file in table_path.glob("*.csv"):
    print(file.name)

print("\nCHARTS:")
for name, paths in charts.items():
    print(name, Path(paths["png"]).exists(), Path(paths["svg"]).exists())

print("\nOK")

IMPORT OK

DATA:
kpi (11, 2)
quality (3, 2)
time_data (168, 3)
severity (2, 2)
top_ip (8, 9)
rules_summary (3, 7)
scenario_summary (4, 7)
anomaly_hours (5, 4)
alert_hours (31, 3)

JSON:
<class 'dict'>
dict_keys(['kpi', 'quality', 'time_data', 'severity', 'top_ip', 'rules_summary', 'scenario_summary', 'anomaly_hours', 'alert_hours'])
[{'metric': 'Liczba logów sieciowych', 'value': '5000'}, {'metric': 'Liczba alertów', 'value': '36'}]

TABLES:
alert_hours.csv
anomaly_hours.csv
kpi_report.csv
rules_summary.csv
scenario_summary.csv
top_suspicious_ips.csv

CHARTS:
logs_alerts_time True True
alerts_severity True True
top_suspicious_ips True True
rules_hits_count True True
rules_alerts_count True True
scenario_logs_count True True
scenario_alerts_count True True
logs_anomalies True True
alert_hours True True

OK
